In [1]:
import os
import polars as pl
from tqdm import tqdm
from tempfile import NamedTemporaryFile
from dataclasses import dataclass


@dataclass 
class SensitivityConfig:
    input_csv: str
    output_dir: str
    score_col: str = "tfidf"
    chunk_size: int = 10_000_000
    skip_if_exists: bool = True
    min_group_size: int = 3  # Minimum members at each SCOP level


class SensitivityCalculator:
    SCOP_LEVELS = ["family", "superfamily", "fold", "class"]

    def __init__(self, config: SensitivityConfig):
        self.config = config
        self.input_csv = os.path.expanduser(config.input_csv)
        self.output_dir = os.path.expanduser(config.output_dir)
        os.makedirs(self.output_dir, exist_ok=True)

        basename = os.path.basename(self.input_csv).replace(".csv", "")
        self.output_pq = os.path.join(
            self.output_dir,
            f"{basename}.sensitivity.{config.score_col}.min{config.min_group_size}.pq"
        )

    def _parse_scop(self, df, prefix, name_col):
        df = df.with_columns([
            pl.col(name_col).str.split(" ").list.get(0).alias(f"{prefix}_scop_id"),
            pl.col(name_col).str.split(" ").list.get(1).alias(f"{prefix}_lineage"),
        ])
        parts = pl.col(f"{prefix}_lineage").str.split(".")
        return df.with_columns([
            parts.list.get(0).alias(f"{prefix}_class"),
            (parts.list.get(0) + pl.lit(".") + parts.list.get(1)).alias(f"{prefix}_fold"),
            (parts.list.get(0) + pl.lit(".") + parts.list.get(1) + pl.lit(".") + parts.list.get(2)).alias(f"{prefix}_superfamily"),
            pl.col(f"{prefix}_lineage").alias(f"{prefix}_family"),
        ])

    def _get_n_counts(self):
        unique = pl.scan_csv(self.input_csv).select("query_name").unique()
        unique = self._parse_scop(unique, "query", "query_name").collect()
        
        n_counts = {}
        n_filtered = {}
        for level in self.SCOP_LEVELS:
            col = f"query_{level}"
            counts = unique.group_by(col).len()
            n_counts[level] = dict(zip(counts[col].to_list(), counts["len"].to_list()))
            # Count how many groups pass the filter
            n_filtered[level] = sum(1 for v in n_counts[level].values() if v >= self.config.min_group_size)
        
        print(f"{len(unique):,} queries")
        print(f"Groups with >= {self.config.min_group_size} members:")
        print(f"  family: {n_filtered['family']:,}/{len(n_counts['family']):,} | "
              f"superfam: {n_filtered['superfamily']:,}/{len(n_counts['superfamily']):,} | "
              f"fold: {n_filtered['fold']:,}/{len(n_counts['fold']):,} | "
              f"class: {n_filtered['class']:,}/{len(n_counts['class']):,}")
        return n_counts

    def _process_chunk(self, chunk, n_counts):
        df = chunk.lazy()
        df = self._parse_scop(df, "query", "query_name")
        df = self._parse_scop(df, "target", "target_name")
        df = df.filter(pl.col("query_md5") != pl.col("target_md5"))
        
        for level in self.SCOP_LEVELS:
            df = df.with_columns([
                (pl.col(f"query_{level}") == pl.col(f"target_{level}")).alias(f"same_{level}"),
                pl.col(f"query_{level}").replace_strict(n_counts[level], default=1).alias(f"n_{level}"),
            ])
        
        df = df.select([
            "query_scop_id", "target_scop_id", self.config.score_col,
            *[f"same_{level}" for level in self.SCOP_LEVELS],
            *[f"n_{level}" for level in self.SCOP_LEVELS],
        ])
        
        return df.group_by(["query_scop_id", "target_scop_id"]).agg([
            pl.col(self.config.score_col).max(),
            *[pl.col(f"same_{level}").first() for level in self.SCOP_LEVELS],
            *[pl.col(f"n_{level}").first() for level in self.SCOP_LEVELS],
        ]).collect()

    def _compute_sensitivity(self, df):
        results = []
        queries = df["query_scop_id"].unique().sort().to_list()
        min_size = self.config.min_group_size
        
        for query_id in tqdm(queries, desc="Computing sensitivity"):
            qdf = df.filter(pl.col("query_scop_id") == query_id).sort(self.config.score_col, descending=True)
            row = {"query_scop_id": query_id}
            
            for level in self.SCOP_LEVELS:
                n_in_group = qdf[f"n_{level}"][0]
                
                # Skip if group too small
                if n_in_group < min_size:
                    row[f"sensitivity_{level}"] = None
                    continue
                
                same_vals = qdf[f"same_{level}"].to_list()
                n_possible = n_in_group - 1  # Exclude self
                
                first_fp = next((i for i, v in enumerate(same_vals) if not v), None)
                
                if first_fp is None:
                    sens = 1.0
                elif first_fp == 0:
                    sens = 0.0
                else:
                    sens = min(first_fp / n_possible, 1.0)
                
                row[f"sensitivity_{level}"] = sens
            results.append(row)
        
        return pl.DataFrame(results)

    def run(self):
        if self.config.skip_if_exists and os.path.exists(self.output_pq):
            print(f"Loading: {self.output_pq}")
            return pl.read_parquet(self.output_pq)

        n_counts = self._get_n_counts()
        
        reader = pl.read_csv_batched(self.input_csv, batch_size=self.config.chunk_size)
        temp_files = []
        
        pbar = tqdm(desc="Reading CSV chunks")
        while (batches := reader.next_batches(1)):
            processed = self._process_chunk(batches[0], n_counts)
            tmp = NamedTemporaryFile(suffix=".parquet", delete=False, dir=self.output_dir)
            processed.write_parquet(tmp.name)
            temp_files.append(tmp.name)
            tmp.close()
            pbar.update(1)
        pbar.close()
        
        print(f"Combining {len(temp_files)} chunks...")
        combined = pl.scan_parquet(temp_files).group_by(["query_scop_id", "target_scop_id"]).agg([
            pl.col(self.config.score_col).max(),
            *[pl.col(f"same_{level}").first() for level in self.SCOP_LEVELS],
            *[pl.col(f"n_{level}").first() for level in self.SCOP_LEVELS],
        ]).collect()
        print(f"{len(combined):,} unique pairs")
        
        result = self._compute_sensitivity(combined)
        
        # Report how many queries have valid sensitivity at each level
        print("Queries with valid sensitivity:")
        for level in self.SCOP_LEVELS:
            valid = result[f"sensitivity_{level}"].drop_nulls().len()
            print(f"  {level}: {valid:,}")
        
        result.write_parquet(self.output_pq)
        print(f"Saved: {self.output_pq}")
        
        for f in temp_files:
            os.unlink(f)
        
        return result

In [2]:
config = SensitivityConfig(
    input_csv="~/data/scope/astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k15.scaled1.kmerseek.results.csv",
    output_dir="~/data/scope/sensitivity_results/",
    score_col="tfidf",
    chunk_size=10_000_000,
    min_group_size=3,  # Only include groups with >= 3 members
)

calc = SensitivityCalculator(config)
results = calc.run()

FileNotFoundError: No such file or directory (os error 2): ...cope/astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k15.scaled1.kmerseek.results.csv (set POLARS_VERBOSE=1 to see full path)

This error occurred with the following context stack:
	[1] 'csv scan'
	[2] 'select'
	[3] 'unique'
	[4] 'with_columns'
	[5] 'with_columns'
	[6] 'sink'


In [ ]:
sensitivity = pl.read_parquet(
    "/Users/olga/data/scope/sensitivity_results/astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k15.scaled1.kmerseek.results.sensitivity.tfidf.min3.pq"
)
sensitivity

query_scop_id,sensitivity_family,sensitivity_superfamily,sensitivity_fold,sensitivity_class
str,f64,f64,f64,f64
"""d12asa_""",0.0,0.0,0.0,0.0
"""d16vpa_""",null,null,null,0.0
"""d1914a1""",null,0.0,0.0,0.0
"""d1914a2""",null,0.0,0.0,0.000274
"""d1a04a1""",0.0,0.0,0.0,0.0
…,…,…,…,…
"""g2dbx.1""",0.0,0.0,0.0,0.0
"""g2vt1.1""",null,0.0,0.0,0.0
"""g3bzy.1""",null,0.0,0.0,0.000274


In [ ]:
sensitivity.describe()

statistic,query_scop_id,sensitivity_family,sensitivity_superfamily,sensitivity_fold,sensitivity_class
str,str,f64,f64,f64,f64
"""count""","""15176""",10827.0,13612.0,14303.0,15176.0
"""null_count""","""0""",4349.0,1564.0,873.0,0.0
"""mean""",null,0.000128,0.000116,0.000099,0.000097
"""std""",null,0.005424,0.002755,0.002452,0.000239
"""min""","""d12asa_""",0.0,0.0,0.0,0.0
"""25%""",null,0.0,0.0,0.0,0.0
"""50%""",null,0.0,0.0,0.0,0.0
"""75%""",null,0.0,0.0,0.0,0.0
"""max""","""g6rlx.1""",0.5,0.2,0.2,0.006601


In [ ]:
# Load the full dataset with all metrics
import numpy as np

# Read a sample to explore relationships
sample = pl.read_csv(
    "~/data/scope/astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k15.scaled1.kmerseek.results.csv",
    n_rows=1_000_000  # Sample first 1M rows for exploration
)

# Add log overlap probability
sample = sample.with_columns([
    pl.col("overlap_probability").log10().alias("log_overlap_prob")
])

sample.select([
    "tfidf", "overlap_probability", "log_overlap_prob", 
    "max_containment", "jaccard"
]).describe()

In [ ]:
# Create scatter plots to visualize relationships
import matplotlib.pyplot as plt
import seaborn as sns

# Sample even further for visualization
viz_sample = sample.sample(n=min(50_000, len(sample)), seed=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. TF-IDF vs log overlap probability
ax = axes[0, 0]
ax.hexbin(viz_sample["log_overlap_prob"], viz_sample["tfidf"], 
          gridsize=50, cmap='viridis', mincnt=1, bins='log')
ax.set_xlabel("Log10(Overlap Probability)")
ax.set_ylabel("TF-IDF")
ax.set_title("TF-IDF vs Log Overlap Probability")
ax.axvline(x=np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='p=0.05')
ax.axvline(x=np.log10(0.01), color='orange', linestyle='--', alpha=0.5, label='p=0.01')
ax.legend()

# 2. Max Containment vs log overlap probability
ax = axes[0, 1]
ax.hexbin(viz_sample["log_overlap_prob"], viz_sample["max_containment"], 
          gridsize=50, cmap='viridis', mincnt=1, bins='log')
ax.set_xlabel("Log10(Overlap Probability)")
ax.set_ylabel("Max Containment")
ax.set_title("Max Containment vs Log Overlap Probability")
ax.axvline(x=np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='p=0.05')
ax.axvline(x=np.log10(0.01), color='orange', linestyle='--', alpha=0.5, label='p=0.01')
ax.legend()

# 3. Jaccard vs log overlap probability
ax = axes[0, 2]
ax.hexbin(viz_sample["log_overlap_prob"], viz_sample["jaccard"], 
          gridsize=50, cmap='viridis', mincnt=1, bins='log')
ax.set_xlabel("Log10(Overlap Probability)")
ax.set_ylabel("Jaccard")
ax.set_title("Jaccard vs Log Overlap Probability")
ax.axvline(x=np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='p=0.05')
ax.axvline(x=np.log10(0.01), color='orange', linestyle='--', alpha=0.5, label='p=0.01')
ax.legend()

# 4. TF-IDF vs Max Containment
ax = axes[1, 0]
ax.hexbin(viz_sample["max_containment"], viz_sample["tfidf"], 
          gridsize=50, cmap='viridis', mincnt=1, bins='log')
ax.set_xlabel("Max Containment")
ax.set_ylabel("TF-IDF")
ax.set_title("TF-IDF vs Max Containment")

# 5. TF-IDF vs Jaccard
ax = axes[1, 1]
ax.hexbin(viz_sample["jaccard"], viz_sample["tfidf"], 
          gridsize=50, cmap='viridis', mincnt=1, bins='log')
ax.set_xlabel("Jaccard")
ax.set_ylabel("TF-IDF")
ax.set_title("TF-IDF vs Jaccard")

# 6. Max Containment vs Jaccard
ax = axes[1, 2]
ax.hexbin(viz_sample["jaccard"], viz_sample["max_containment"], 
          gridsize=50, cmap='viridis', mincnt=1, bins='log')
ax.set_xlabel("Jaccard")
ax.set_ylabel("Max Containment")
ax.set_title("Max Containment vs Jaccard")

plt.tight_layout()
plt.show()

In [ ]:
# Analyze distribution of overlap probabilities
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of overlap probabilities
ax = axes[0]
ax.hist(sample["overlap_probability"], bins=100, edgecolor='black', alpha=0.7)
ax.set_xlabel("Overlap Probability")
ax.set_ylabel("Count")
ax.set_title("Distribution of Overlap Probabilities")
ax.axvline(x=0.05, color='red', linestyle='--', linewidth=2, label='p=0.05')
ax.axvline(x=0.01, color='orange', linestyle='--', linewidth=2, label='p=0.01')
ax.axvline(x=0.001, color='green', linestyle='--', linewidth=2, label='p=0.001')
ax.legend()
ax.set_yscale('log')

# Histogram of log overlap probabilities
ax = axes[1]
ax.hist(sample["log_overlap_prob"], bins=100, edgecolor='black', alpha=0.7)
ax.set_xlabel("Log10(Overlap Probability)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Log10 Overlap Probabilities")
ax.axvline(x=np.log10(0.05), color='red', linestyle='--', linewidth=2, label='p=0.05')
ax.axvline(x=np.log10(0.01), color='orange', linestyle='--', linewidth=2, label='p=0.01')
ax.axvline(x=np.log10(0.001), color='green', linestyle='--', linewidth=2, label='p=0.001')
ax.legend()

plt.tight_layout()
plt.show()

# Show statistics on different cutoffs
print("Statistics on overlap probability cutoffs:")
for cutoff in [0.001, 0.01, 0.05, 0.1]:
    n_below = (sample["overlap_probability"] < cutoff).sum()
    pct_below = 100 * n_below / len(sample)
    print(f"  p < {cutoff}: {n_below:,} ({pct_below:.2f}%)")

In [ ]:
# Parse SCOP information to see how metrics vary by true/false positives
def parse_scop_simple(df):
    """Parse SCOP lineages for query and target"""
    df = df.with_columns([
        pl.col("query_name").str.split(" ").list.get(1).alias("query_lineage"),
        pl.col("target_name").str.split(" ").list.get(1).alias("target_lineage"),
    ])
    
    # Extract family (full lineage)
    df = df.with_columns([
        pl.col("query_lineage").alias("query_family"),
        pl.col("target_lineage").alias("target_family"),
    ])
    
    # Check if same family
    df = df.with_columns([
        (pl.col("query_family") == pl.col("target_family")).alias("same_family")
    ])
    
    return df

sample_with_scop = parse_scop_simple(sample)

# Analyze metrics for true positives vs false positives
true_pos = sample_with_scop.filter(pl.col("same_family"))
false_pos = sample_with_scop.filter(~pl.col("same_family"))

print(f"True positives (same family): {len(true_pos):,}")
print(f"False positives (different family): {len(false_pos):,}")
print(f"Ratio: {len(false_pos)/len(true_pos):.2f}:1 (FP:TP)")
print()

# Compare metrics
metrics = ["tfidf", "max_containment", "jaccard", "overlap_probability"]
print("Metric comparisons (median values):")
print(f"{'Metric':<25} {'True Pos':<15} {'False Pos':<15} {'Ratio (TP/FP)':<15}")
print("-" * 70)
for metric in metrics:
    tp_median = true_pos[metric].median()
    fp_median = false_pos[metric].median()
    ratio = tp_median / fp_median if fp_median > 0 else float('inf')
    print(f"{metric:<25} {tp_median:<15.6f} {fp_median:<15.6f} {ratio:<15.2f}")

In [ ]:
# Visualize metric distributions for TP vs FP with different overlap probability cutoffs
cutoffs = [0.001, 0.01, 0.05, 1.0]  # 1.0 = no cutoff

fig, axes = plt.subplots(len(cutoffs), 3, figsize=(15, 4*len(cutoffs)))

for i, cutoff in enumerate(cutoffs):
    # Filter by overlap probability cutoff
    if cutoff < 1.0:
        filtered = sample_with_scop.filter(pl.col("overlap_probability") < cutoff)
        title_suffix = f" (p < {cutoff})"
    else:
        filtered = sample_with_scop
        title_suffix = " (no filter)"
    
    tp = filtered.filter(pl.col("same_family"))
    fp = filtered.filter(~pl.col("same_family"))
    
    # TF-IDF
    ax = axes[i, 0]
    if len(tp) > 0:
        ax.hist(tp["tfidf"], bins=50, alpha=0.5, label=f'TP (n={len(tp):,})', color='green', density=True)
    if len(fp) > 0:
        ax.hist(fp["tfidf"], bins=50, alpha=0.5, label=f'FP (n={len(fp):,})', color='red', density=True)
    ax.set_xlabel("TF-IDF")
    ax.set_ylabel("Density")
    ax.set_title(f"TF-IDF Distribution{title_suffix}")
    ax.legend()
    ax.set_xlim(0, min(500, sample_with_scop["tfidf"].max()))
    
    # Max Containment
    ax = axes[i, 1]
    if len(tp) > 0:
        ax.hist(tp["max_containment"], bins=50, alpha=0.5, label=f'TP (n={len(tp):,})', color='green', density=True)
    if len(fp) > 0:
        ax.hist(fp["max_containment"], bins=50, alpha=0.5, label=f'FP (n={len(fp):,})', color='red', density=True)
    ax.set_xlabel("Max Containment")
    ax.set_ylabel("Density")
    ax.set_title(f"Max Containment Distribution{title_suffix}")
    ax.legend()
    
    # Jaccard
    ax = axes[i, 2]
    if len(tp) > 0:
        ax.hist(tp["jaccard"], bins=50, alpha=0.5, label=f'TP (n={len(tp):,})', color='green', density=True)
    if len(fp) > 0:
        ax.hist(fp["jaccard"], bins=50, alpha=0.5, label=f'FP (n={len(fp):,})', color='red', density=True)
    ax.set_xlabel("Jaccard")
    ax.set_ylabel("Density")
    ax.set_title(f"Jaccard Distribution{title_suffix}")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Calculate precision/recall-like metrics for different cutoffs
print("Effect of overlap probability cutoffs on TP/FP ratio:")
print(f"{'Cutoff':<15} {'Total Pairs':<15} {'TP':<15} {'FP':<15} {'TP%':<10} {'FP%':<10} {'TP/FP Ratio':<15}")
print("-" * 95)

for cutoff in [1.0, 0.1, 0.05, 0.01, 0.001, 0.0001]:
    if cutoff < 1.0:
        filtered = sample_with_scop.filter(pl.col("overlap_probability") < cutoff)
    else:
        filtered = sample_with_scop
    
    n_total = len(filtered)
    n_tp = filtered.filter(pl.col("same_family")).shape[0]
    n_fp = filtered.filter(~pl.col("same_family")).shape[0]
    
    tp_pct = 100 * n_tp / n_total if n_total > 0 else 0
    fp_pct = 100 * n_fp / n_total if n_total > 0 else 0
    ratio = n_tp / n_fp if n_fp > 0 else float('inf')
    
    cutoff_str = f"p < {cutoff}" if cutoff < 1.0 else "No filter"
    print(f"{cutoff_str:<15} {n_total:<15,} {n_tp:<15,} {n_fp:<15,} {tp_pct:<10.2f} {fp_pct:<10.2f} {ratio:<15.4f}")

In [ ]:
# Create correlation matrix
correlation_df = sample.select([
    "tfidf", "max_containment", "jaccard", "overlap_probability", "log_overlap_prob"
])

# Calculate correlation matrix
import pandas as pd
corr_matrix = correlation_df.to_pandas().corr()

# Plot correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation Matrix of Metrics", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey insights:")
print("- TF-IDF correlation with max_containment:", corr_matrix.loc['tfidf', 'max_containment'])
print("- TF-IDF correlation with jaccard:", corr_matrix.loc['tfidf', 'jaccard'])
print("- TF-IDF correlation with log_overlap_prob:", corr_matrix.loc['tfidf', 'log_overlap_prob'])
print("- Max containment correlation with jaccard:", corr_matrix.loc['max_containment', 'jaccard'])
print("- Overlap probability correlation with max_containment:", corr_matrix.loc['overlap_probability', 'max_containment'])

## Publication-Quality Figures (FoldSeek/TEA style)

In [ ]:
# Set publication-quality plotting style
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['figure.titlesize'] = 12
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']

# Define colors for different metrics/methods
COLORS = {
    'kmerseek_tfidf': '#E74C3C',  # Red
    'kmerseek_jaccard': '#3498DB',  # Blue  
    'kmerseek_containment': '#F39C12',  # Orange
    'mmseqs2': '#F4D03F',  # Yellow
    'blast': '#E59866',  # Light orange
    'foldseek': '#9B59B6',  # Purple
}

In [ ]:
# Figure A: Sensitivity curve (like FoldSeek Fig 1a, TEA Fig 2A)
# Shows "Sensitivity up to 1st FP" vs "Fraction of queries"

def compute_sensitivity_curve(df, score_col, scop_level="family"):
    """
    Compute sensitivity curve: for each query, calculate sensitivity 
    (fraction of true positives before first false positive)
    """
    # Parse SCOP lineages
    df = df.with_columns([
        pl.col("query_name").str.split(" ").list.get(1).alias("query_lineage"),
        pl.col("target_name").str.split(" ").list.get(1).alias("target_lineage"),
    ])
    
    # Extract SCOP level
    if scop_level == "family":
        df = df.with_columns([
            pl.col("query_lineage").alias("query_scop"),
            pl.col("target_lineage").alias("target_scop"),
        ])
    elif scop_level == "superfamily":
        parts_q = pl.col("query_lineage").str.split(".")
        parts_t = pl.col("target_lineage").str.split(".")
        df = df.with_columns([
            (parts_q.list.get(0) + pl.lit(".") + parts_q.list.get(1) + pl.lit(".") + parts_q.list.get(2)).alias("query_scop"),
            (parts_t.list.get(0) + pl.lit(".") + parts_t.list.get(1) + pl.lit(".") + parts_t.list.get(2)).alias("target_scop"),
        ])
    elif scop_level == "fold":
        parts_q = pl.col("query_lineage").str.split(".")
        parts_t = pl.col("target_lineage").str.split(".")
        df = df.with_columns([
            (parts_q.list.get(0) + pl.lit(".") + parts_q.list.get(1)).alias("query_scop"),
            (parts_t.list.get(0) + pl.lit(".") + parts_t.list.get(1)).alias("target_scop"),
        ])
    
    df = df.with_columns([
        (pl.col("query_scop") == pl.col("target_scop")).alias("same_scop")
    ])
    
    # Remove self-hits
    df = df.filter(pl.col("query_md5") != pl.col("target_md5"))
    
    # Get unique queries
    queries = df["query_name"].unique().to_list()
    
    sensitivities = []
    for query in queries:
        qdf = df.filter(pl.col("query_name") == query).sort(score_col, descending=True)
        
        if len(qdf) == 0:
            continue
            
        same_vals = qdf["same_scop"].to_list()
        
        # Count total positives for this query
        n_positives = sum(same_vals)
        
        if n_positives == 0:
            continue
        
        # Find first false positive
        first_fp = next((i for i, v in enumerate(same_vals) if not v), None)
        
        if first_fp is None:
            # No false positives - retrieved all positives
            sensitivity = 1.0
        elif first_fp == 0:
            # First hit is FP
            sensitivity = 0.0
        else:
            # Sensitivity = fraction of positives retrieved before first FP
            sensitivity = min(first_fp / n_positives, 1.0)
        
        sensitivities.append(sensitivity)
    
    # Sort sensitivities descending
    sensitivities = sorted(sensitivities, reverse=True)
    
    # Create fraction of queries
    fractions = np.arange(len(sensitivities)) / len(sensitivities)
    
    return fractions, sensitivities

# Test with sample data
print("Computing sensitivity curves...")
frac_tfidf, sens_tfidf = compute_sensitivity_curve(sample, "tfidf", scop_level="superfamily")
frac_jaccard, sens_jaccard = compute_sensitivity_curve(sample, "jaccard", scop_level="superfamily")
frac_containment, sens_containment = compute_sensitivity_curve(sample, "max_containment", scop_level="superfamily")

print(f"TF-IDF: {len(sens_tfidf)} queries")
print(f"Jaccard: {len(sens_jaccard)} queries")  
print(f"Max Containment: {len(sens_containment)} queries")

In [ ]:
# Plot Figure A: Sensitivity curves (TEA-style)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Panel A: Family-level sensitivity
ax = axes[0]
frac_fam_tfidf, sens_fam_tfidf = compute_sensitivity_curve(sample, "tfidf", scop_level="family")
frac_fam_jaccard, sens_fam_jaccard = compute_sensitivity_curve(sample, "jaccard", scop_level="family")
frac_fam_cont, sens_fam_cont = compute_sensitivity_curve(sample, "max_containment", scop_level="family")

ax.plot(frac_fam_tfidf, sens_fam_tfidf, '-', color=COLORS['kmerseek_tfidf'], 
        linewidth=2, label='KmerSeek (TF-IDF)', marker='o', markersize=3, markevery=0.1)
ax.plot(frac_fam_jaccard, sens_fam_jaccard, '-', color=COLORS['kmerseek_jaccard'], 
        linewidth=2, label='KmerSeek (Jaccard)', marker='s', markersize=3, markevery=0.1)
ax.plot(frac_fam_cont, sens_fam_cont, '-', color=COLORS['kmerseek_containment'], 
        linewidth=2, label='KmerSeek (Max Containment)', marker='^', markersize=3, markevery=0.1)

ax.set_xlabel('Fraction of queries', fontweight='bold')
ax.set_ylabel('Sensitivity up to the 1st FP', fontweight='bold')
ax.set_title('Family Sensitivity', fontweight='bold', loc='left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(frameon=True, fancybox=False, edgecolor='black')
ax.text(-0.15, 1.05, 'A', transform=ax.transAxes, fontsize=16, fontweight='bold')

# Panel B: Superfamily-level sensitivity  
ax = axes[1]
ax.plot(frac_tfidf, sens_tfidf, '-', color=COLORS['kmerseek_tfidf'], 
        linewidth=2, label='KmerSeek (TF-IDF)', marker='o', markersize=3, markevery=0.1)
ax.plot(frac_jaccard, sens_jaccard, '-', color=COLORS['kmerseek_jaccard'], 
        linewidth=2, label='KmerSeek (Jaccard)', marker='s', markersize=3, markevery=0.1)
ax.plot(frac_containment, sens_containment, '-', color=COLORS['kmerseek_containment'], 
        linewidth=2, label='KmerSeek (Max Containment)', marker='^', markersize=3, markevery=0.1)

ax.set_xlabel('Fraction of queries', fontweight='bold')
ax.set_ylabel('Sensitivity up to the 1st FP', fontweight='bold')
ax.set_title('Superfamily Sensitivity', fontweight='bold', loc='left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(frameon=True, fancybox=False, edgecolor='black')
ax.text(-0.15, 1.05, 'B', transform=ax.transAxes, fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('sensitivity_curves.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure B: Query Coverage (like FoldSeek Fig 1d, TEA Fig 2E)
# Shows query coverage vs TP hits up to 1st FP

def compute_query_coverage(df, score_col, scop_level="family"):
    """
    For each query, compute:
    - Number of TP hits before first FP
    - Query coverage (fraction of query covered by hits)
    """
    # Parse SCOP
    df = df.with_columns([
        pl.col("query_name").str.split(" ").list.get(1).alias("query_lineage"),
        pl.col("target_name").str.split(" ").list.get(1).alias("target_lineage"),
    ])
    
    # Extract SCOP level
    if scop_level == "family":
        df = df.with_columns([
            pl.col("query_lineage").alias("query_scop"),
            pl.col("target_lineage").alias("target_scop"),
        ])
    elif scop_level == "superfamily":
        parts_q = pl.col("query_lineage").str.split(".")
        parts_t = pl.col("target_lineage").str.split(".")
        df = df.with_columns([
            (parts_q.list.get(0) + pl.lit(".") + parts_q.list.get(1) + pl.lit(".") + parts_q.list.get(2)).alias("query_scop"),
            (parts_t.list.get(0) + pl.lit(".") + parts_t.list.get(1) + pl.lit(".") + parts_t.list.get(2)).alias("target_scop"),
        ])
    
    df = df.with_columns([
        (pl.col("query_scop") == pl.col("target_scop")).alias("same_scop")
    ])
    
    # Remove self-hits
    df = df.filter(pl.col("query_md5") != pl.col("target_md5"))
    
    queries = df["query_name"].unique().to_list()
    
    tp_hits = []
    coverages = []
    
    for query in queries:
        qdf = df.filter(pl.col("query_name") == query).sort(score_col, descending=True)
        
        if len(qdf) == 0:
            continue
        
        same_vals = qdf["same_scop"].to_list()
        
        # Find first FP
        first_fp = next((i for i, v in enumerate(same_vals) if not v), len(same_vals))
        
        # Get TPs before first FP
        tp_count = sum(same_vals[:first_fp])
        
        # Calculate coverage (assuming coverage from region_length or similar)
        # For now, use simple fraction
        if first_fp > 0:
            coverage = min(first_fp / (len(same_vals) + 1), 1.0)
        else:
            coverage = 0.0
        
        tp_hits.append(tp_count)
        coverages.append(coverage)
    
    return tp_hits, coverages

# Compute for different metrics
tp_tfidf, cov_tfidf = compute_query_coverage(sample, "tfidf", scop_level="superfamily")
tp_jaccard, cov_jaccard = compute_query_coverage(sample, "jaccard", scop_level="superfamily")
tp_cont, cov_cont = compute_query_coverage(sample, "max_containment", scop_level="superfamily")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

ax.scatter(tp_tfidf, cov_tfidf, alpha=0.3, s=10, color=COLORS['kmerseek_tfidf'], 
           label='KmerSeek (TF-IDF)', rasterized=True)
ax.scatter(tp_jaccard, cov_jaccard, alpha=0.3, s=10, color=COLORS['kmerseek_jaccard'], 
           label='KmerSeek (Jaccard)', rasterized=True)
ax.scatter(tp_cont, cov_cont, alpha=0.3, s=10, color=COLORS['kmerseek_containment'], 
           label='KmerSeek (Max Containment)', rasterized=True)

ax.set_xlabel('TP hits up to 1st FP', fontweight='bold')
ax.set_ylabel('Query coverage', fontweight='bold')
ax.set_title('Query Coverage vs TP Hits', fontweight='bold', loc='left')
ax.set_xlim(0, max(max(tp_tfidf), max(tp_jaccard), max(tp_cont)) * 1.1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(frameon=True, fancybox=False, edgecolor='black')
ax.text(-0.15, 1.05, 'C', transform=ax.transAxes, fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('query_coverage.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure C: ROC-style curve with overlap probability filtering
# Similar to FoldSeek weighted ROC

def compute_precision_recall(df, score_col, scop_level="superfamily", overlap_cutoff=1.0):
    """
    Compute precision-recall curve with optional overlap probability filtering
    """
    # Parse SCOP
    df = df.with_columns([
        pl.col("query_name").str.split(" ").list.get(1).alias("query_lineage"),
        pl.col("target_name").str.split(" ").list.get(1).alias("target_lineage"),
    ])
    
    # Extract SCOP level
    if scop_level == "superfamily":
        parts_q = pl.col("query_lineage").str.split(".")
        parts_t = pl.col("target_lineage").str.split(".")
        df = df.with_columns([
            (parts_q.list.get(0) + pl.lit(".") + parts_q.list.get(1) + pl.lit(".") + parts_q.list.get(2)).alias("query_scop"),
            (parts_t.list.get(0) + pl.lit(".") + parts_t.list.get(1) + pl.lit(".") + parts_t.list.get(2)).alias("target_scop"),
        ])
    
    df = df.with_columns([
        (pl.col("query_scop") == pl.col("target_scop")).alias("is_positive")
    ])
    
    # Remove self-hits
    df = df.filter(pl.col("query_md5") != pl.col("target_md5"))
    
    # Filter by overlap probability
    if overlap_cutoff < 1.0:
        df = df.filter(pl.col("overlap_probability") < overlap_cutoff)
    
    # Sort by score descending
    df = df.sort(score_col, descending=True)
    
    # Compute cumulative TP and FP
    is_pos = df["is_positive"].to_numpy()
    
    tp_cumsum = np.cumsum(is_pos)
    fp_cumsum = np.cumsum(~is_pos)
    
    total_positives = is_pos.sum()
    
    # Precision and Recall
    precision = tp_cumsum / (tp_cumsum + fp_cumsum)
    recall = tp_cumsum / total_positives
    
    # Subsample for plotting
    indices = np.unique(np.linspace(0, len(precision)-1, min(1000, len(precision))).astype(int))
    
    return recall[indices], precision[indices]

# Compute PR curves with different overlap probability cutoffs
print("Computing precision-recall curves...")
cutoffs = [1.0, 0.05, 0.01, 0.001]
pr_curves = {}

for cutoff in cutoffs:
    recall, precision = compute_precision_recall(sample, "tfidf", scop_level="superfamily", overlap_cutoff=cutoff)
    pr_curves[cutoff] = (recall, precision)
    print(f"  p < {cutoff}: {len(recall)} points")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

colors = ['#E74C3C', '#F39C12', '#27AE60', '#3498DB']
for i, cutoff in enumerate(cutoffs):
    recall, precision = pr_curves[cutoff]
    label = f'No filter' if cutoff >= 1.0 else f'p < {cutoff}'
    ax.plot(recall, precision, '-', linewidth=2.5, label=label, color=colors[i])

ax.set_xlabel('Recall', fontweight='bold')
ax.set_ylabel('Precision', fontweight='bold')
ax.set_title('Superfamily Precision-Recall\n(TF-IDF with overlap probability filtering)', 
             fontweight='bold', loc='left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(frameon=True, fancybox=False, edgecolor='black', title='Overlap prob. cutoff')
ax.text(-0.15, 1.05, 'D', transform=ax.transAxes, fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('precision_recall_overlap_filter.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure D: Multi-panel combined figure (like TEA Figure 2)
# Combines sensitivity curves, AUC comparison, and coverage plots

def compute_auc(fractions, sensitivities):
    """Compute area under sensitivity curve using trapezoidal rule"""
    from scipy.integrate import trapezoid
    return trapezoid(sensitivities, fractions)

# Compute AUCs for different metrics and SCOP levels
metrics = {
    'TF-IDF': 'tfidf',
    'Jaccard': 'jaccard', 
    'Max Containment': 'max_containment'
}

levels = ['family', 'superfamily', 'fold']

auc_results = {level: {} for level in levels}

for level in levels:
    for name, col in metrics.items():
        frac, sens = compute_sensitivity_curve(sample, col, scop_level=level)
        auc = compute_auc(frac, sens)
        auc_results[level][name] = auc
        print(f"{level} {name}: AUC = {auc:.4f}")

# Create multi-panel figure
fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.35)

# Panel A: Family sensitivity
ax = fig.add_subplot(gs[0, 0])
for name, col in metrics.items():
    frac, sens = compute_sensitivity_curve(sample, col, scop_level="family")
    color = COLORS.get(f'kmerseek_{col}', '#333333')
    ax.plot(frac, sens, '-', linewidth=2, label=name, color=color)

ax.set_xlabel('Fraction of queries', fontweight='bold')
ax.set_ylabel('Sensitivity up to the 1st FP', fontweight='bold')
ax.set_title('Family', fontweight='bold', loc='left', fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(frameon=True, fancybox=False, edgecolor='black', fontsize=8)
ax.text(-0.25, 1.08, 'A', transform=ax.transAxes, fontsize=14, fontweight='bold')

# Panel B: Superfamily sensitivity
ax = fig.add_subplot(gs[0, 1])
for name, col in metrics.items():
    frac, sens = compute_sensitivity_curve(sample, col, scop_level="superfamily")
    color = COLORS.get(f'kmerseek_{col}', '#333333')
    ax.plot(frac, sens, '-', linewidth=2, label=name, color=color)

ax.set_xlabel('Fraction of queries', fontweight='bold')
ax.set_ylabel('Sensitivity up to the 1st FP', fontweight='bold')
ax.set_title('Superfamily', fontweight='bold', loc='left', fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(frameon=True, fancybox=False, edgecolor='black', fontsize=8)
ax.text(-0.25, 1.08, 'B', transform=ax.transAxes, fontsize=14, fontweight='bold')

# Panel C: AUC comparison
ax = fig.add_subplot(gs[0, 2])
x_pos = np.arange(len(levels))
width = 0.25

for i, (name, col) in enumerate(metrics.items()):
    aucs = [auc_results[level][name] for level in levels]
    color = COLORS.get(f'kmerseek_{col}', '#333333')
    ax.bar(x_pos + i*width, aucs, width, label=name, color=color, edgecolor='black', linewidth=0.5)

ax.set_xlabel('SCOP Level', fontweight='bold')
ax.set_ylabel('Area Under Curve', fontweight='bold')
ax.set_title('AUC Comparison', fontweight='bold', loc='left', fontsize=11)
ax.set_xticks(x_pos + width)
ax.set_xticklabels(['Family', 'Superfam', 'Fold'], fontsize=9)
ax.set_ylim(0, max([max(auc_results[l].values()) for l in levels]) * 1.1)
ax.grid(True, alpha=0.3, linestyle='--', axis='y')
ax.legend(frameon=True, fancybox=False, edgecolor='black', fontsize=8)
ax.text(-0.25, 1.08, 'C', transform=ax.transAxes, fontsize=14, fontweight='bold')

# Panel D: Overlap probability impact on precision-recall
ax = fig.add_subplot(gs[1, :2])
cutoffs = [1.0, 0.05, 0.01, 0.001]
colors_pr = ['#E74C3C', '#F39C12', '#27AE60', '#3498DB']

for i, cutoff in enumerate(cutoffs):
    recall, precision = compute_precision_recall(sample, "tfidf", overlap_cutoff=cutoff)
    label = f'No filter' if cutoff >= 1.0 else f'p < {cutoff}'
    ax.plot(recall, precision, '-', linewidth=2, label=label, color=colors_pr[i])

ax.set_xlabel('Recall', fontweight='bold')
ax.set_ylabel('Precision', fontweight='bold')
ax.set_title('Effect of Overlap Probability Filtering\n(Superfamily, TF-IDF)', 
             fontweight='bold', loc='left', fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(frameon=True, fancybox=False, edgecolor='black', title='Overlap prob.', fontsize=8)
ax.text(-0.12, 1.08, 'D', transform=ax.transAxes, fontsize=14, fontweight='bold')

# Panel E: Query coverage
ax = fig.add_subplot(gs[1, 2])
for name, col in metrics.items():
    tp_hits, coverages = compute_query_coverage(sample, col, scop_level="superfamily")
    color = COLORS.get(f'kmerseek_{col}', '#333333')
    ax.scatter(tp_hits, coverages, alpha=0.4, s=8, color=color, label=name, rasterized=True)

ax.set_xlabel('TP hits up to 1st FP', fontweight='bold')
ax.set_ylabel('Query coverage', fontweight='bold')
ax.set_title('Coverage vs TP Hits', fontweight='bold', loc='left', fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(frameon=True, fancybox=False, edgecolor='black', fontsize=8)
ax.text(-0.25, 1.08, 'E', transform=ax.transAxes, fontsize=14, fontweight='bold')

plt.savefig('combined_benchmark_figure.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics table for overlap probability filtering
# Similar to tables in FoldSeek/TEA papers

print("Summary: Impact of Overlap Probability Filtering on Performance")
print("=" * 100)
print(f"\n{'Metric':<20} {'No filter':<15} {'p < 0.05':<15} {'p < 0.01':<15} {'p < 0.001':<15}")
print("-" * 100)

for name, col in metrics.items():
    aucs = []
    for cutoff in [1.0, 0.05, 0.01, 0.001]:
        # Filter data
        if cutoff < 1.0:
            filtered = sample.filter(pl.col("overlap_probability") < cutoff)
        else:
            filtered = sample
        
        # Compute AUC
        frac, sens = compute_sensitivity_curve(filtered, col, scop_level="superfamily")
        auc = compute_auc(frac, sens)
        aucs.append(auc)
    
    print(f"{name:<20} {aucs[0]:<15.4f} {aucs[1]:<15.4f} {aucs[2]:<15.4f} {aucs[3]:<15.4f}")

print("\n" + "=" * 100)
print("\nRecommendation: Based on precision-recall curves and AUC values,")
print("an overlap probability cutoff of p < 0.01 provides a good balance between")
print("filtering random matches while retaining true positive signal.")

## Status and verdict

**Largely unexecuted, and what did run found nothing.** The first cell raises `FileNotFoundError`,
17 of the 20 code cells have no output, and the publication-figure section at the end never ran.

The one real result is the per-query sensitivity table over 15,176 SCOPe40 queries, and it is
effectively all zero: mean family sensitivity 0.000128, mean superfamily 0.000116, mean fold
0.000099, with the median at 0.0 at every level and the best single query reaching only 0.5 at
family level. Null counts are large as well (4,349 queries have no family-level value, 1,564 no
superfamily), because those queries have no same-family or same-superfamily peer in the set.

That is a negative result, and it is consistent with what
[053](./053-scope40-hp-k15-plots.ipynb) shows in detail on the same data: HP at this k-size on the
full SCOPe40 benchmark does not rank homologs above non-homologs. Nothing here needs rerunning as
written. The question it was meant to answer is covered by 053 and, with the coverage denominator
handled properly, by [067](./067_scope_all_metrics_all_fps.ipynb).